In [1]:
import json
import os

# Feeder network structure: Substation -> Feeders -> Transformers -> Villages
# Using the SAME village names already referenced in the ML component's
# integration docs, so both pieces line up without renaming anything later.

feeder_network = {
    "nodes": [
        {"id": "SUB1", "type": "substation", "name": "Main Substation", "lat": 18.55, "lon": 73.75},
        {"id": "FDR1", "type": "feeder", "name": "Feeder Line A", "lat": 18.53, "lon": 73.78},
        {"id": "FDR2", "type": "feeder", "name": "Feeder Line B", "lat": 18.58, "lon": 73.80},
        {"id": "TXR1", "type": "transformer", "name": "Transformer T1", "lat": 18.51, "lon": 73.80},
        {"id": "TXR2", "type": "transformer", "name": "Transformer T2", "lat": 18.60, "lon": 73.82},
        {"id": "VIL1", "type": "village", "name": "Village Rampur", "lat": 18.50, "lon": 73.81},
        {"id": "VIL2", "type": "village", "name": "Village Shivpur", "lat": 18.61, "lon": 73.83},
    ],
    "edges": [
        {"source": "SUB1", "target": "FDR1"},
        {"source": "SUB1", "target": "FDR2"},
        {"source": "FDR1", "target": "TXR1"},
        {"source": "FDR2", "target": "TXR2"},
        {"source": "TXR1", "target": "VIL1"},
        {"source": "TXR2", "target": "VIL2"},
    ]
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/feeder_network.json', 'w') as f:
    json.dump(feeder_network, f, indent=2)

print(f"Feeder network saved: {len(feeder_network['nodes'])} nodes, {len(feeder_network['edges'])} edges")
for node in feeder_network['nodes']:
    print(f"  {node['id']:6s} ({node['type']:12s}) — {node['name']}")

Feeder network saved: 7 nodes, 6 edges
  SUB1   (substation  ) — Main Substation
  FDR1   (feeder      ) — Feeder Line A
  FDR2   (feeder      ) — Feeder Line B
  TXR1   (transformer ) — Transformer T1
  TXR2   (transformer ) — Transformer T2
  VIL1   (village     ) — Village Rampur
  VIL2   (village     ) — Village Shivpur


In [2]:
fault_signatures = {
    "conductor_damage": {
        "description": "Broken or damaged conductor wire",
        "voltage_pattern": "Sudden voltage drop to near-zero on affected section",
        "current_pattern": "Current drops sharply or spikes momentarily then drops",
        "typical_cause": "Physical wire snap, corrosion, storm damage"
    },
    "transformer_overload": {
        "description": "Transformer operating beyond rated capacity",
        "voltage_pattern": "Gradual voltage sag over sustained period",
        "current_pattern": "Current consistently above rated threshold",
        "typical_cause": "Excess simultaneous load, summer peak demand"
    },
    "vegetation_contact": {
        "description": "Tree branch or vegetation touching live line",
        "voltage_pattern": "Intermittent voltage fluctuation/flicker",
        "current_pattern": "Irregular current spikes, often weather-correlated",
        "typical_cause": "Untrimmed trees near feeder lines, monsoon growth"
    },
    "illegal_connection": {
        "description": "Unauthorized power tapping",
        "voltage_pattern": "Localized voltage drop in specific section",
        "current_pattern": "Current higher than expected for registered load",
        "typical_cause": "Unmetered rural tapping"
    }
}

with open('../outputs/fault_signatures.json', 'w') as f:
    json.dump(fault_signatures, f, indent=2)

print(f"Saved {len(fault_signatures)} fault type signatures:")
for k, v in fault_signatures.items():
    print(f"  - {k}: {v['description']}")

Saved 4 fault type signatures:
  - conductor_damage: Broken or damaged conductor wire
  - transformer_overload: Transformer operating beyond rated capacity
  - vegetation_contact: Tree branch or vegetation touching live line
  - illegal_connection: Unauthorized power tapping


In [3]:
import networkx as nx

# Build the graph from the feeder_network data we already defined
G = nx.Graph()

for node in feeder_network['nodes']:
    G.add_node(node['id'], **node)

for edge in feeder_network['edges']:
    G.add_edge(edge['source'], edge['target'])

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Nodes: {list(G.nodes())}")
print(f"Edges: {list(G.edges())}")

Graph built: 7 nodes, 6 edges
Nodes: ['SUB1', 'FDR1', 'FDR2', 'TXR1', 'TXR2', 'VIL1', 'VIL2']
Edges: [('SUB1', 'FDR1'), ('SUB1', 'FDR2'), ('FDR1', 'TXR1'), ('FDR2', 'TXR2'), ('TXR1', 'VIL1'), ('TXR2', 'VIL2')]


In [4]:
# Simulate: a fault was detected near Village Rampur (VIL1)
# Find the shortest inspection path from the Substation to reach it

target_village = "VIL1"  # Village Rampur
path = nx.shortest_path(G, source="SUB1", target=target_village)

print(f"\nFault reported near: {G.nodes[target_village]['name']}")
print(f"Recommended inspection path from Main Substation:")
print("  " + " → ".join(G.nodes[n]['name'] for n in path))
print(f"\nPath length: {len(path)-1} hops")


Fault reported near: Village Rampur
Recommended inspection path from Main Substation:
  Main Substation → Feeder Line A → Transformer T1 → Village Rampur

Path length: 3 hops


In [5]:
# Load the ACTUAL fault_alerts.json produced by the ML component
ml_output_path = '../../ml-anomaly-detection/outputs/fault_alerts.json'

with open(ml_output_path, 'r') as f:
    fault_alerts = json.load(f)

print(f"Loaded {len(fault_alerts)} fault alerts from ML component")
for alert in fault_alerts:
    print(f"  {alert['timestamp']} | Ia={alert['current_Ia']} | village={alert['village']} | fault_type={alert['fault_type']}")

Loaded 5 fault alerts from ML component
  2026-08-17 14:00:00 | Ia=729.85 | village=None | fault_type=unclassified
  2026-08-17 14:05:00 | Ia=502.85 | village=None | fault_type=unclassified
  2026-08-17 14:10:00 | Ia=-36.0 | village=None | fault_type=unclassified
  2026-08-17 14:15:00 | Ia=-16.11 | village=None | fault_type=unclassified
  2026-08-17 14:20:00 | Ia=-813.94 | village=None | fault_type=unclassified


In [6]:
import random
random.seed(42)

villages = [n for n in feeder_network['nodes'] if n['type'] == 'village']

enriched_alerts = []
for alert in fault_alerts:
    # Assign to a village (in a real system this would come from which
    # sensor/transformer reported the reading — here we assign for demo purposes)
    assigned_village = random.choice(villages)

    # Compute the recommended inspection path from substation to that village
    path = nx.shortest_path(G, source="SUB1", target=assigned_village['id'])
    path_names = [G.nodes[n]['name'] for n in path]

    enriched = alert.copy()
    enriched['village'] = assigned_village['name']
    enriched['recommended_inspection_path'] = path_names
    enriched['inspection_hops'] = len(path) - 1
    enriched_alerts.append(enriched)

with open('../outputs/enriched_fault_alerts.json', 'w') as f:
    json.dump(enriched_alerts, f, indent=2)

print(f"Saved {len(enriched_alerts)} enriched fault alerts\n")
print(json.dumps(enriched_alerts[0], indent=2))

Saved 5 enriched fault alerts

{
  "timestamp": "2026-08-17 14:00:00",
  "current_Ia": 729.85,
  "voltage_Va": -0.004,
  "fault_type": "unclassified",
  "anomaly_score": -0.0293,
  "status": "Fault Detected",
  "village": "Village Rampur",
  "recommended_inspection_path": [
    "Main Substation",
    "Feeder Line A",
    "Transformer T1",
    "Village Rampur"
  ],
  "inspection_hops": 3
}


In [7]:
# Load the FULL 300-entry timeseries (not just the 5 sampled fault_alerts)
timeseries_path = '../../ml-anomaly-detection/outputs/timeseries_readings.json'

with open(timeseries_path, 'r') as f:
    full_timeseries = json.load(f)

print(f"Loaded {len(full_timeseries)} sequential readings from ML component")
print(f"Faults in this stream: {sum(1 for r in full_timeseries if r['status']=='Fault Detected')}")

Loaded 300 sequential readings from ML component
Faults in this stream: 13


In [8]:
random.seed(42)  # same seed as before, keeps village assignment consistent

enriched_timeseries = []
for reading in full_timeseries:
    enriched = reading.copy()
    if reading['status'] == 'Fault Detected':
        assigned_village = random.choice(villages)
        path = nx.shortest_path(G, source="SUB1", target=assigned_village['id'])
        enriched['village'] = assigned_village['name']
        enriched['recommended_inspection_path'] = [G.nodes[n]['name'] for n in path]
        enriched['inspection_hops'] = len(path) - 1
    else:
        enriched['village'] = None
        enriched['recommended_inspection_path'] = None
        enriched['inspection_hops'] = None
    enriched_timeseries.append(enriched)

with open('../outputs/enriched_timeseries.json', 'w') as f:
    json.dump(enriched_timeseries, f, indent=2)

print(f"Saved {len(enriched_timeseries)} enriched readings to outputs/enriched_timeseries.json")
print(f"\nSample fault entry:")
sample_fault = next(r for r in enriched_timeseries if r['status'] == 'Fault Detected')
print(json.dumps(sample_fault, indent=2))

Saved 300 enriched readings to outputs/enriched_timeseries.json

Sample fault entry:
{
  "timestamp": "2026-08-18 09:00:00",
  "current_Ia": -170.47,
  "voltage_Va": 0.0545,
  "status": "Fault Detected",
  "anomaly_score": -0.0557,
  "village": "Village Rampur",
  "recommended_inspection_path": [
    "Main Substation",
    "Feeder Line A",
    "Transformer T1",
    "Village Rampur"
  ],
  "inspection_hops": 3
}


In [9]:
import time
from IPython.display import clear_output

N = 30  # same pattern as ML — bump to 300 later once you're ready for the full run

for i, reading in enumerate(enriched_timeseries[:N]):
    clear_output(wait=True)
    if reading['status'] == 'Fault Detected':
        print(f"[{reading['timestamp']}] 🔴 FAULT — {reading['village']}")
        print(f"   Route: {' → '.join(reading['recommended_inspection_path'])}")
    else:
        print(f"[{reading['timestamp']}] 🟢 Normal")
    time.sleep(0.15)

[2026-08-18 09:04:50] 🟢 Normal


In [10]:
import json

with open('../outputs/enriched_fault_alerts.json') as f:
    small = json.load(f)
with open('../outputs/enriched_timeseries.json') as f:
    full = json.load(f)

print(f"enriched_fault_alerts.json: {len(small)} entries")
print(f"enriched_timeseries.json: {len(full)} entries")

enriched_fault_alerts.json: 5 entries
enriched_timeseries.json: 300 entries
